# 🛣️ RoadSense — YOLOv8 Pothole Vision Retraining Pipeline (GPU Accelerated)

This notebook trains a high-precision YOLOv8 computer vision model for road damage and pothole detection.

**Hardware Recommendation:** 
- On Kaggle: Turn ON **Accelerator -> GPU T4 x2** (Free)
- On Google Colab: Turn ON **Runtime -> Change runtime type -> T4 GPU** (Free)

Estimated training time: ~25 - 35 minutes for 50 epochs.

In [ ]:
# 1. Install & Verify Dependencies
!pip install -q ultralytics roboflow

import torch
from ultralytics import YOLO
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Download Annotated Pothole Dataset
We use a pre-annotated benchmark dataset of real road dashcam frames with 80/20 train/val splits and augmentation.

In [ ]:
# Download high-quality pothole benchmark dataset via Roboflow Universe
import os
from roboflow import Roboflow

# Public benchmark road damage dataset (YOLOv8 format)
!curl -L -o pothole_dataset.zip "https://universe.roboflow.com/ds/8uJ1q3E02Z?key=Z5p26d83yO"
!unzip -q pothole_dataset.zip -d ./pothole_dataset
!rm pothole_dataset.zip

print("Dataset downloaded & extracted successfully!")
print("Contents:", os.listdir('./pothole_dataset'))

## 3. Train YOLOv8 Model on GPU
We use `yolov8n.pt` (nano) or `yolov8s.pt` (small). The small variant provides higher feature resolution for small or shallow potholes with minimal inference latency.

In [ ]:
# Initialize model with pretrained COCO vision weights
model = YOLO('yolov8s.pt') # 'yolov8s.pt' for high accuracy or 'yolov8n.pt' for ultra-lightweight

# Train on GPU
results = model.train(
    data='./pothole_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    # Data augmentations tuned for road textures
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    project='RoadSense_YOLO',
    name='pothole_detector_v2'
)

## 4. Evaluate Validation Metrics (mAP & Confusion Matrix)

In [ ]:
# Validate the newly trained model
metrics = model.val()
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 5. Download the Upgraded Weights for RoadSense
Run the cell below to download `pothole_yolov8.pt`. Once downloaded, simply place it into:
`RoadSense/main/cv_pipeline/pothole_yolov8.pt`

In [ ]:
import shutil
from google.colab import files

best_weights = 'RoadSense_YOLO/pothole_detector_v2/weights/best.pt'
target_name = 'pothole_yolov8.pt'

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_name)
    print(f"Saved as {target_name} ({os.path.getsize(target_name)/(1024*1024):.2f} MB)")
    try:
        files.download(target_name)
    except Exception:
        print(f"If on Kaggle, download '{target_name}' from the Output panel on the right.")
else:
    print("Weights not found, please check training run directory.")